# Qwen2.5-3B-Instruct — Kaggle T4×2 Serving
## OpenAI-compatible REST API · cloudflared public tunnel · optional API key

| Setting | Value |
|---------|-------|
| Model | `Qwen/Qwen2.5-3B-Instruct` |
| GPUs | 2 × T4 (16 GB each) via `device_map="auto"` |
| Endpoint style | OpenAI `v1/chat/completions` |
| Tunnel | cloudflare quick tunnel (no account needed) |
| Auth | Optional bearer-token API key |

> **Swap model:** change `MODEL_NAME` in Cell 2 to any HF model slug,
> e.g. `"Qwen/Qwen2.5-7B-Instruct"` or `"Qwen/Qwen3-4B"`.


In [ ]:
# Install serving stack (transformers already on Kaggle; add fastapi + uvicorn)
!pip install -q fastapi uvicorn[standard] pydantic accelerate


In [ ]:
# ── Configuration — edit these before running ────────────────────────────────

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # HF model slug
PORT       = 8000                           # uvicorn port

# API key auth (leave empty string "" to serve without authentication)
API_KEY    = ""   # e.g. "my-secret-key-42"

DTYPE      = "float16"   # float16 fits 3B comfortably on one T4; bfloat16 also fine
MAX_NEW_TOKENS_DEFAULT = 512


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import torch, os, time, re, threading, subprocess, textwrap
from datetime import datetime

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPUs     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i} : {props.name}  {props.total_memory/1e9:.1f} GB")
print(f"dtype    : {DTYPE}")
print(f"model    : {MODEL_NAME}")
print(f"auth     : {'enabled (API_KEY set)' if API_KEY else 'disabled (open access)'}")
print("=" * 60)


In [ ]:
# Authenticate with HuggingFace using the Kaggle Secret named HF_TOKEN.
# Add it under: Notebook → Add-ons → Secrets → Add New Secret  (name: HF_TOKEN)
# Qwen2.5-3B-Instruct is NOT gated, so the token is optional —
# but it prevents anonymous rate-limits for large model downloads.

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace: authenticated via Kaggle Secret")
except Exception as e:
    print(f"HuggingFace: running without token ({e})")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model    : {MODEL_NAME} ...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DTYPE == "float16" else torch.bfloat16,
    device_map="auto",          # spreads across both T4s automatically
    trust_remote_code=True,
)
model.eval()

elapsed = time.time() - t0
total_params = sum(p.numel() for p in model.parameters()) / 1e9

print(f"\nModel loaded in {elapsed:.1f}s")
print(f"Parameters      : {total_params:.2f}B")
print(f"Device map      : {model.hf_device_map if hasattr(model, 'hf_device_map') else 'single device'}")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i} VRAM     : {used:.2f} / {total:.1f} GB used")


In [ ]:
from fastapi import FastAPI, HTTPException, Depends, Security, Request
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uuid

app = FastAPI(
    title="Qwen2.5-3B Inference API",
    description="OpenAI-compatible /v1/chat/completions endpoint",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

# ── Auth ──────────────────────────────────────────────────────────────────────
_bearer = HTTPBearer(auto_error=False)

def require_auth(creds: HTTPAuthorizationCredentials = Security(_bearer)):
    if not API_KEY:
        return   # auth disabled
    if creds is None or creds.credentials != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing API key")

# ── Request / response models ─────────────────────────────────────────────────
class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = MODEL_NAME
    messages: list[Message]
    max_tokens: int = MAX_NEW_TOKENS_DEFAULT
    temperature: float = 0.7
    top_p: float = 0.9
    do_sample: Optional[bool] = None   # auto-set from temperature

class Delta(BaseModel):
    role: str
    content: str

# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    gpu_info = [
        {"gpu": i,
         "name": torch.cuda.get_device_properties(i).name,
         "vram_used_gb": round(torch.cuda.memory_allocated(i)/1e9, 2),
         "vram_total_gb": round(torch.cuda.get_device_properties(i).total_memory/1e9, 1)}
        for i in range(torch.cuda.device_count())
    ]
    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "auth_required": bool(API_KEY),
        "gpus": gpu_info,
        "timestamp": datetime.utcnow().isoformat() + "Z",
    }

@app.get("/v1/models", dependencies=[Depends(require_auth)])
def list_models():
    return {
        "object": "list",
        "data": [{"id": MODEL_NAME, "object": "model", "owned_by": "user"}],
    }

@app.post("/v1/chat/completions", dependencies=[Depends(require_auth)])
def chat_completions(req: ChatRequest):
    # Build prompt with the model's own chat template
    messages = [{"role": m.role, "content": m.content} for m in req.messages]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    # Clamp temperature — if 0 use greedy
    temp = max(req.temperature, 0.0)
    do_sample = req.do_sample if req.do_sample is not None else (temp > 0)

    generate_kwargs = dict(
        **inputs,
        max_new_tokens=req.max_tokens,
        top_p=req.top_p,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        generate_kwargs["temperature"] = temp

    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(**generate_kwargs)
    latency_ms = (time.perf_counter() - t0) * 1000

    new_tokens = output_ids[0][prompt_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    completion_tokens = len(new_tokens)
    total_tokens      = prompt_len + completion_tokens

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": answer},
            "finish_reason": "stop",
            "latency_ms": round(latency_ms, 1),
        }],
        "usage": {
            "prompt_tokens": prompt_len,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "tokens_per_sec": round(completion_tokens / (latency_ms / 1000), 1),
        },
    }

print("FastAPI app defined — endpoints:")
print("  GET  /health")
print("  GET  /v1/models")
print("  POST /v1/chat/completions")
if API_KEY:
    print(f"  Auth : Bearer token required")
else:
    print(f"  Auth : disabled (open access)")


In [ ]:
import uvicorn

# ── 1. Start uvicorn in a background thread ───────────────────────────────────
def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(3)   # let uvicorn bind
print(f"Server : http://localhost:{PORT}  (background thread running)")

# ── 2. Download cloudflared binary ───────────────────────────────────────────
cf_bin = "/kaggle/working/cloudflared"
if not os.path.exists(cf_bin):
    print("Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q", "-O", cf_bin,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    subprocess.run(["chmod", "+x", cf_bin], check=True)
    print("cloudflared downloaded")

# ── 3. Start the quick tunnel and capture public URL ─────────────────────────
cf_proc = subprocess.Popen(
    [cf_bin, "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

public_url = None
deadline = time.time() + 40
while time.time() < deadline:
    line = cf_proc.stderr.readline()
    if not line:
        time.sleep(0.1)
        continue
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("\n" + "=" * 60)
    print("PUBLIC URL (share this):")
    print(f"  {public_url}")
    print()
    print("Health check:")
    print(f"  {public_url}/health")
    print()
    print("Chat endpoint:")
    print(f"  POST  {public_url}/v1/chat/completions")
    if API_KEY:
        print(f"  Header: Authorization: Bearer {API_KEY}")
    else:
        print("  No auth required")
    print("=" * 60)
else:
    print("Could not capture cloudflared URL — check cf_proc.stderr manually")
    print(cf_proc.stderr.read()[:500])


In [ ]:
# Quick local test (runs in this kernel)
import requests

headers = {}
if API_KEY:
    headers["Authorization"] = f"Bearer {API_KEY}"

payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "You are a helpful travel-industry assistant."},
        {"role": "user",   "content": "Explain PNR structure in two sentences."},
    ],
    "max_tokens": 150,
    "temperature": 0.7,
}

resp = requests.post(f"http://localhost:{PORT}/v1/chat/completions",
                     json=payload, headers=headers, timeout=120)
resp.raise_for_status()
data = resp.json()

print("Response:")
print(data["choices"][0]["message"]["content"])
print()
print("Usage:", data["usage"])


## Usage examples

Replace `<PUBLIC_URL>` with the URL printed by Cell 7.

### curl (no auth)
```bash
curl -X POST <PUBLIC_URL>/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "messages": [{"role":"user","content":"Hello! Who are you?"}],
    "max_tokens": 200
  }'
```

### curl (with API key)
```bash
curl -X POST <PUBLIC_URL>/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer my-secret-key-42" \
  -d '{
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "messages": [{"role":"user","content":"Hello!"}],
    "max_tokens": 200
  }'
```

### Python openai client (drop-in)
```python
from openai import OpenAI

client = OpenAI(
    base_url="<PUBLIC_URL>/v1",
    api_key="my-secret-key-42",   # any string if auth disabled
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[{"role": "user", "content": "Explain PNR structure."}],
    max_tokens=300,
)
print(response.choices[0].message.content)
```

### Stop the tunnel
```python
cf_proc.terminate()
```
